# Willow connectivity graph

Qubits that are available (keys of `sq_pauli_error_rate`) and CZ pairs that are allowed (keys of `cz_pauli_error_rate`) form a graph on the Willow 105 grid. `plot_willow` draws it in the spec-sheet layout: column across, row down, one square per qubit.

In [ ]:
# %pip install cirq cirq-google matplotlib networkx
import itertools, math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import networkx as nx
import cirq, cirq_google

DEVICE = cirq_google.Willow105.metadata
ALL_QUBITS = sorted(DEVICE.qubit_set)
ALL_PAIRS = sorted({frozenset(p) for p in DEVICE.qubit_pairs}, key=lambda p: sorted(p))
print(f"{len(ALL_QUBITS)} qubits, {len(ALL_PAIRS)} couplers, "
      f"mean degree {2 * len(ALL_PAIRS) / len(ALL_QUBITS):.2f}")

In [ ]:
def plot_willow(qubits=None, pairs=None, *, qubit_values=None, pair_values=None,
                qubit_label="", pair_label="", highlight=None, readout_lines=False,
                show_ids=False, title="", ax=None):
    # qubits / pairs: what is available (default: whole device). Missing ones are drawn as dead.
    # qubit_values: {GridQubit: float}; pair_values: {(q0, q1) or frozenset: float}.
    # highlight: set of qubits to outline (e.g. a candidate patch for one instance).
    qubits = set(ALL_QUBITS if qubits is None else qubits)
    pairs = {frozenset(p) for p in (ALL_PAIRS if pairs is None else pairs)}
    pv = {frozenset(k): (v[0] if isinstance(v, tuple) else v) for k, v in (pair_values or {}).items()}
    qv = {k: (v[0] if isinstance(v, tuple) else v) for k, v in (qubit_values or {}).items()}
    highlight = set(highlight or ())
    ax = ax or plt.subplots(figsize=(9, 8))[1]
    xy = lambda q: (q.col, q.row)

    if readout_lines:  # qubits with equal row + col share a readout line
        for d in range(min(q.row + q.col for q in ALL_QUBITS), max(q.row + q.col for q in ALL_QUBITS) + 1):
            line = sorted((q for q in ALL_QUBITS if q.row + q.col == d), key=lambda q: q.col)
            if line:
                ax.plot([q.col for q in line], [q.row for q in line], lw=14, alpha=0.15,
                        color="C0" if d % 2 else "C1", solid_capstyle="round", zorder=0)
                ax.text(line[0].col - 0.6, line[0].row + 0.6, str(d), fontsize=7, color="0.4", ha="center")

    # couplers: dead (dotted grey), alive (coloured by value or plain)
    dead = [p for p in ALL_PAIRS if p not in pairs]
    ax.add_collection(LineCollection([[xy(a), xy(b)] for a, b in map(sorted, dead)],
                                     colors="0.75", linestyles=":", lw=1, zorder=1))
    alive = [p for p in ALL_PAIRS if p in pairs]
    segs = [[xy(a), xy(b)] for a, b in map(sorted, alive)]
    widths = [(7 if p <= highlight else 4.5) if pv else (4.5 if p <= highlight else 2.5) for p in alive]
    size = 90 if pv else 150
    if pv:
        lc = LineCollection(segs, cmap="viridis", lw=widths, zorder=2)
        lc.set_array(np.array([pv.get(p, np.nan) for p in alive]))
        ax.add_collection(lc)
        plt.colorbar(lc, ax=ax, shrink=0.6, pad=0.02, label=pair_label)
    else:
        ax.add_collection(LineCollection(segs, colors="0.45", lw=widths, zorder=2))

    # qubits: dead (grey x), alive (squares, coloured by value or plain)
    for q in ALL_QUBITS:
        if q not in qubits:
            ax.scatter(*xy(q), marker="x", s=60, c="0.6", zorder=3)
    live = [q for q in ALL_QUBITS if q in qubits]
    edge = ["crimson" if q in highlight else "0.2" for q in live]
    lw = [2.5 if q in highlight else 0.6 for q in live]
    if qv:
        sc = ax.scatter([q.col for q in live], [q.row for q in live], marker="s", s=size,
                        c=[qv.get(q, np.nan) for q in live], cmap="magma_r",
                        edgecolors=edge, linewidths=lw, zorder=4)
        plt.colorbar(sc, ax=ax, shrink=0.6, pad=0.08, label=qubit_label)
    else:
        ax.scatter([q.col for q in live], [q.row for q in live], marker="s", s=size,
                   c="white", edgecolors=edge, linewidths=lw, zorder=4)
    if show_ids:
        for q in live:
            ax.text(q.col, q.row, f"{q.row},{q.col}", fontsize=4.5, ha="center", va="center", zorder=5)

    rows = range(min(q.row for q in ALL_QUBITS), max(q.row for q in ALL_QUBITS) + 1)
    cols = range(min(q.col for q in ALL_QUBITS), max(q.col for q in ALL_QUBITS) + 1)
    ax.set_xticks(cols); ax.set_yticks(rows)
    ax.xaxis.tick_top(); ax.set_xlabel("column"); ax.xaxis.set_label_position("top")
    ax.set_ylabel("row"); ax.tick_params(labelsize=7, length=0)
    ax.set_xlim(min(cols) - 1, max(cols) + 1); ax.set_ylim(max(rows) + 1, min(rows) - 1)
    ax.set_aspect("equal"); [s.set_visible(False) for s in ax.spines.values()]
    ax.set_title(f"{title}\n{len(live)} qubits, {len(alive)} CZ pairs", fontsize=11)
    return ax


def to_networkx(sq_pauli_error_rate, cz_pauli_error_rate):
    # Available-hardware graph with error rates as node / edge attributes.
    g = nx.Graph()
    for q, (e, _) in sq_pauli_error_rate.items():
        g.add_node(q, sq_error=e)
    for (a, b), (e, _) in cz_pauli_error_rate.items():
        if a in g and b in g:
            g.add_edge(a, b, cz_error=e)
    return g

## Example 1: the full Willow 105 device

In [ ]:
plot_willow(show_ids=True, title="Willow 105 (cirq_google.Willow105)");

## Example 2: a fake calibration with dropouts

Same generators as the challenge notebook. Dead qubits are grey ×, dead couplers dotted. Squares are coloured by expected readout error for a half-filled state, ½·e₀ + ½·e₁; edges by CZ Pauli error.

In [ ]:
def pick_single_qubit_dropouts(dropout_rate, rng):
    all_qubits = np.array(list(DEVICE.qubit_set))
    return set(all_qubits[rng.random(len(all_qubits)) < dropout_rate])

def generate_qubit_error_rates(median, spread, dropouts, uncertainty, rng):
    working = [q for q in DEVICE.qubit_set if q not in dropouts]
    rates = rng.lognormal(np.log(median), spread, len(working))
    return {q: (r, uncertainty) for q, r in zip(working, rates)}

def generate_two_qubit_pauli_error_rates(rng, single_qubit_dropouts, median=2.5e-3, spread=0.5,
                                         dropout_rate=0.01, uncertainty=5e-4):
    working = [tuple(p) for p in DEVICE.qubit_pairs
               if all(q not in single_qubit_dropouts for q in p) and rng.random() > dropout_rate]
    rates = rng.lognormal(np.log(median), spread, len(working))
    return {p: (r, uncertainty) for p, r in zip(working, rates)}

rng = np.random.default_rng(7)
dropouts = pick_single_qubit_dropouts(0.03, rng)          # 3% so a few show up
sq_pauli_error_rate = generate_qubit_error_rates(4e-4, 0.5, dropouts, 2e-4, rng)
cz_pauli_error_rate = generate_two_qubit_pauli_error_rates(rng, dropouts, dropout_rate=0.03)
zero_errors = generate_qubit_error_rates(5e-3, 0.5, dropouts, 2e-4, rng)
one_errors = generate_qubit_error_rates(1e-2, 0.5, dropouts, 3e-4, rng)

readout_cost = {q: 0.5 * zero_errors[q][0] + 0.5 * one_errors[q][0] for q in zero_errors}
plot_willow(sq_pauli_error_rate, cz_pauli_error_rate,
            qubit_values=readout_cost, pair_values=cz_pauli_error_rate,
            qubit_label="readout error, ½e₀ + ½e₁", pair_label="CZ Pauli error",
            title="Available graph from a fake calibration");

## Example 3: readout lines (bonus question)

Qubits with equal row + col share a readout line, and a mid-circuit measurement must cover a whole line. The bands are those diagonals.

In [ ]:
plot_willow(sq_pauli_error_rate, cz_pauli_error_rate, readout_lines=True,
            title="Readout lines (row + col = const)");

## Example 4: a candidate patch for one instance

A deliberately naive greedy placer for D(8,4): start at the best qubit, then repeatedly add the neighbour with the lowest CZ error plus readout cost. It only shows how to highlight a patch; a real placer should search.

In [ ]:
def greedy_patch(g, N, n, zero_errors, one_errors):
    ro = lambda q: (N - n) / N * zero_errors[q][0] + n / N * one_errors[q][0]
    patch = {min(g.nodes, key=lambda q: ro(q) + min((g.edges[q, r]["cz_error"] for r in g[q]), default=1))}
    while len(patch) < N:
        frontier = {(r, q) for q in patch for r in g[q] if r not in patch}
        r, _ = min(frontier, key=lambda e: g.edges[e]["cz_error"] + ro(e[0]))
        patch.add(r)
    return patch

g = to_networkx(sq_pauli_error_rate, cz_pauli_error_rate)
patch = greedy_patch(g, 8, 4, zero_errors, one_errors)
print("patch:", sorted(patch))
plot_willow(sq_pauli_error_rate, cz_pauli_error_rate, pair_values=cz_pauli_error_rate,
            pair_label="CZ Pauli error", highlight=patch, title="Greedy patch for D(8,4)");